# Day 042 — XGBoost Architecture & Practice: Understanding from Scratch (Math + Code + API)

> **Style:** Every concept is first explained in plain English, then reinforced with mathematics (2nd-order Taylor expansion, Similarity Score), followed by custom Python calculations, and finally hands-on practice with the official `xgboost` API.  
> **Goal:** Develop a deep, thorough understanding of XGBoost's internal architecture — leaving no room for confusion!

---

## Part 1 — What Is XGBoost and How Does It Differ from Standard Gradient Boosting?

**XGBoost** stands for **eXtreme Gradient Boosting**. It is an optimized, scalable implementation of gradient-boosted decision trees (GBDT) designed for speed and performance. While the core idea — sequentially adding weak learners (trees) that correct the errors of the previous ensemble — is shared with standard Gradient Boosting Machines (GBM), XGBoost introduces several key innovations that make it significantly faster, more accurate, and more robust against overfitting.

The table below summarizes the most important differences:

| Feature | Standard Gradient Boosting (GBM) | XGBoost |
| :--- | :--- | :--- |
| **Optimization** | Uses only 1st-order derivatives (Gradient $g_i$) | Uses a **2nd-order Taylor Expansion** (both Gradient $g_i$ and Hessian $h_i$), enabling a more accurate quadratic approximation of the loss |
| **Overfitting Control** | Relies primarily on shrinkage (learning rate) | **Built-in L1 ($\\alpha$) & L2 ($\\lambda$) regularization** on leaf weights, plus tree-complexity penalty ($\\gamma$) for pruning |
| **Split Finding** | Exact Greedy (evaluates every possible split point) | **Exact Greedy + Approximate Quantile Sketch + Histogram-based** methods for handling large datasets efficiently |
| **Missing Values** | Requires manual imputation before training | **Sparsity-aware split finding** — handles missing values natively by learning the optimal default direction during training |
| **Speed & Hardware** | Typically sequential, single-threaded on CPU | **Parallelized feature-level tree building, cache-aware access patterns, out-of-core computation, and GPU support** |
| **System Design** | Basic implementation | Column-block structure for efficient sorting, cache-aware prefetching, and support for distributed computing |

### Why These Differences Matter

- **2nd-order information (Hessian)** gives XGBoost a better sense of the *curvature* of the loss function, allowing it to take more informed steps when fitting each new tree.
- **Built-in regularization** means you get overfitting protection *by default*, rather than relying solely on external techniques like early stopping.
- **Approximate split finding** allows XGBoost to scale to datasets with millions of rows where exact greedy search would be prohibitively expensive.
- **Native missing-value handling** saves preprocessing effort and often yields better results than arbitrary imputation.

---

## Part 2 — Our Example Dataset

To build intuition, we will work through a tiny, easy-to-follow dataset of **5 houses**. Each house has a single feature — its **Area (in square feet)** — and a target variable — its **Price** (in ₹ Lakh).

| House | Area (sq.ft) | Price ($y$ in ₹ Lakh) |
| :---: | :---: | :---: |
| A | 600 | 15 |
| B | 900 | 22 |
| C | 1200 | 30 |
| D | 1600 | 40 |
| E | 2000 | 50 |

We set the **initial base prediction** $\\hat{y}_i^{(0)} = 0.5$ for all samples. In practice, XGBoost's default initial prediction is often the mean of the target values (for regression) or a log-odds value (for classification), but we use 0.5 here for simplicity so we can clearly see how large the initial residuals are.

> **Note:** The choice of initial prediction only affects the first boosting round. The algorithm will quickly correct for it in subsequent rounds.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Dataset setup
X = np.array([600, 900, 1200, 1600, 2000]).reshape(-1, 1)
y = np.array([15.0, 22.0, 30.0, 40.0, 50.0])

df = pd.DataFrame({'Area': X.ravel(), 'Price (y)': y})
print("Dataset:")
print(df.to_string(index=False))

## Part 3 — Math Deep Dive: Regularized Objective & 2nd-Order Taylor Expansion

This is the mathematical heart of XGBoost. Understanding this section will clarify *why* XGBoost works the way it does.

### 1. The Objective Function

At boosting step $t$, our total objective function consists of two parts — the **training loss** and a **regularization penalty**:

$$\\mathcal{O}^{(t)} = \\sum_{i=1}^n l\\left(y_i, \\hat{y}_i^{(t-1)} + f_t(x_i)\\right) + \\Omega(f_t)$$

Where:
- $l(\\cdot)$ is any differentiable loss function (e.g., MSE for regression, log-loss for classification)
- $\\hat{y}_i^{(t-1)}$ is the prediction from the ensemble built so far (up to step $t-1$)
- $f_t(x_i)$ is the output of the **new tree** being added at step $t$
- $\\Omega(f_t)$ is the **regularization penalty** on the new tree

The **regularization term** $\\Omega(f_t)$ is defined as:

$$\\Omega(f_t) = \\gamma T + \\frac{1}{2}\\lambda \\sum_{j=1}^T w_j^2 + \\alpha \\sum_{j=1}^T |w_j|$$

Where:
- $T$ = total number of leaves in tree $f_t$ (penalizes complex trees)
- $w_j$ = the predicted score (weight) assigned to leaf $j$
- $\\gamma$ = **minimum loss reduction** required to make a further split (tree complexity penalty). A higher $\\gamma$ makes the algorithm more conservative.
- $\\lambda$ = **L2 regularization** coefficient on leaf weights. Encourages smaller, smoother leaf weights.
- $\\alpha$ = **L1 regularization** coefficient on leaf weights. Encourages sparsity (some leaf weights pushed to zero).

> **Key Insight:** This regularization is what sets XGBoost apart from standard GBM. By directly penalizing tree complexity and leaf magnitude in the objective, XGBoost achieves better generalization.

---

### 2. Taylor Expansion (2nd-Order Approximation)

Directly optimizing the objective above is difficult because $f_t(x_i)$ is the output of a tree (a complex, non-smooth function). XGBoost's key trick is to **approximate the loss using a 2nd-order Taylor expansion**.

Recall that for any smooth function $f(x + \\Delta x)$, the 2nd-order Taylor approximation around $x$ is:

$$f(x + \\Delta x) \\approx f(x) + f'(x)\\Delta x + \\frac{1}{2}f''(x)(\\Delta x)^2$$

In our setting:
- $x = \\hat{y}_i^{(t-1)}$ (the current prediction from the existing ensemble)
- $\\Delta x = f_t(x_i)$ (the output of the new tree we are adding)

We define the **gradient** and **hessian** of the loss with respect to the current prediction:

- **First Derivative (Gradient):**
  $$g_i = \\frac{\\partial \, l(y_i, \\hat{y}_i^{(t-1)})}{\\partial \, \\hat{y}_i^{(t-1)}}$$

- **Second Derivative (Hessian):**
  $$h_i = \\frac{\\partial^2 \, l(y_i, \\hat{y}_i^{(t-1)})}{\\partial \, (\\hat{y}_i^{(t-1)})^2}$$

#### Concrete Examples for Common Loss Functions

**For MSE Loss** ($l = \\frac{1}{2}(y_i - \\hat{y}_i)^2$):
- $g_i = \\hat{y}_i - y_i = -(y_i - \\hat{y}_i)$ — this is simply the negative residual
- $h_i = 1$ — the second derivative is constant, meaning the loss surface is a simple parabola

**For Log-Loss / Binary Cross-Entropy** (used in classification):
- $g_i = p_i - y_i$ — where $p_i = \\sigma(\\hat{y}_i)$ is the predicted probability
- $h_i = p_i(1 - p_i)$ — this is the variance of a Bernoulli distribution, meaning the hessian is highest when the model is most uncertain ($p \\approx 0.5$)

#### The Simplified Objective

After applying the Taylor expansion and dropping the constant term $l(y_i, \\hat{y}_i^{(t-1)})$ (which does not depend on the new tree), the objective simplifies to:

$$\\tilde{\\mathcal{O}}^{(t)} \\approx \\sum_{i=1}^n \\left[ g_i \, f_t(x_i) + \\frac{1}{2} h_i \, f_t(x_i)^2 \\right] + \\gamma T + \\frac{1}{2}\\lambda \\sum_{j=1}^T w_j^2$$

> **Why is this powerful?** This approximation converts the problem of fitting a tree into a *quadratic optimization* problem that has a clean, closed-form solution — as we will see next.

In [ ]:
# Calculate the Gradient (g_i) and Hessian (h_i) for MSE Loss
# Initial prediction: y_pred_0 = 0.5 for all samples
y_pred_0 = np.full_like(y, 0.5)

# Gradient: g_i = y_pred - y = -(y - y_pred)
# A negative gradient means the prediction is too low (model needs to increase its output)
g = y_pred_0 - y

# Hessian: h_i = 1 for MSE (the second derivative of 0.5*(y-yhat)^2 w.r.t. yhat is 1)
h = np.ones_like(y)

df['g_i (Gradient)'] = g
df['h_i (Hessian)'] = h
print("Gradients and Hessians for Step 1:")
print(df.to_string(index=False))
print("\nObservation: All gradients are negative because our initial prediction (0.5)")
print("is far below all true prices. The new tree needs to predict positive values to correct this.")

## Part 4 — Optimal Leaf Weight ($w_j^*$) & Similarity Score

Now we arrive at one of the most elegant results in XGBoost: the closed-form solution for the optimal leaf weight and the quality score for each node.

### Grouping by Leaves

Since all samples in the same leaf $j$ receive the same prediction $w_j$, we can group terms in the objective by leaf. Let $I_j$ denote the set of sample indices that fall into leaf $j$. Then define the **aggregate gradient** and **aggregate hessian** for leaf $j$:

$$G_j = \\sum_{i \\in I_j} g_i, \\quad H_j = \\sum_{i \\in I_j} h_i$$

The simplified objective, rewritten at the leaf level, becomes:

$$\\tilde{\\mathcal{O}}^{(t)} = \\sum_{j=1}^T \\left[ G_j \, w_j + \\frac{1}{2}(H_j + \\lambda) \, w_j^2 \\right] + \\gamma T$$

Notice how the L2 regularization $\\lambda$ gets absorbed into the coefficient of $w_j^2$.

### Deriving the Optimal Leaf Weight $w_j^*$

For each leaf $j$, we want to find the value of $w_j$ that minimizes the objective. Since each leaf's contribution is an independent quadratic in $w_j$, we simply take the derivative with respect to $w_j$, set it to zero, and solve:

$$\\frac{\\partial \\tilde{\\mathcal{O}}}{\\partial w_j} = G_j + (H_j + \\lambda) \, w_j = 0$$

$$\\implies \\boxed{w_j^* = -\\frac{G_j}{H_j + \\lambda}}$$

> **Interpretation:** The optimal leaf weight is the (negative) ratio of the sum of gradients to the sum of hessians (plus regularization). For MSE, where $h_i = 1$, this simplifies to the negative mean gradient in that leaf — essentially the mean residual, dampened by $\\lambda$.

### The Similarity Score (Optimal Objective Value)

Substituting $w_j^*$ back into the objective gives us the **minimum achievable loss** for a given tree structure. The quality of a node (or leaf) is measured by the **Similarity Score**:

$$\\text{Similarity Score} = \\frac{G_j^2}{H_j + \\lambda}$$

A higher similarity score indicates that the samples in the node are more "similar" (their gradients are more aligned), making it a better candidate for a leaf.

### The Split Gain Formula

When we split a parent node $P$ into a left child $L$ and a right child $R$, the **Gain** quantifies how much the objective improves:

$$\\boxed{\\text{Gain} = \\frac{1}{2} \\left[ \\frac{G_L^2}{H_L + \\lambda} + \\frac{G_R^2}{H_R + \\lambda} - \\frac{G_P^2}{H_P + \\lambda} \\right] - \\gamma}$$

- The first three terms compare the sum of the children's similarity scores to the parent's similarity score.
- $\\gamma$ is subtracted as the cost of adding one more split (increasing tree complexity).
- **A split is made only if Gain > 0.** If the gain is negative, the split is not worth the added complexity, and the node remains a leaf (this is how $\\gamma$ drives **pre-pruning**).

In [ ]:
def similarity_score(G, H, reg_lambda=0.0):
    """Calculate the Similarity Score for a node.
    Higher values indicate more 'agreement' among gradients in the node."""
    return (G**2) / (H + reg_lambda)

def optimal_weight(G, H, reg_lambda=0.0):
    """Calculate the optimal leaf weight (prediction) for a node."""
    return -G / (H + reg_lambda)

# Root node metrics (all 5 samples in one node)
G_root = np.sum(g)
H_root = np.sum(h)
reg_lambda = 0.0  # No regularization for this initial demo

sim_root = similarity_score(G_root, H_root, reg_lambda)
w_root = optimal_weight(G_root, H_root, reg_lambda)

print(f"Root Node: G_root = {G_root}, H_root = {H_root}")
print(f"Root Similarity Score = {sim_root:.2f}")
print(f"Root Optimal Weight w* = {w_root:.2f}")
print(f"\nInterpretation: If we made NO splits and predicted a single value for all houses,")
print(f"the optimal prediction would be {w_root:.2f} (close to the mean price of {y.mean():.1f}).")

## Part 5 — Custom Calculation: Finding the Best Split

Now let's put the theory into practice. We will manually evaluate several candidate split points for the **Area** feature and compute the **Gain** for each one. The split with the highest gain is the best split.

We consider four candidate thresholds: **750, 1050, 1400, and 1800**. For each threshold, samples with `Area < threshold` go to the left child, and the rest go to the right child.

This time we set $\\lambda = 1.0$ (L2 regularization) to observe how regularization affects the scores.

In [ ]:
reg_lambda = 1.0  # L2 regularization
gamma = 0.0       # No split penalty (so we can see raw gain values)

split_candidates = [750, 1050, 1400, 1800]

results = []
for split in split_candidates:
    left_mask = X.ravel() < split
    right_mask = ~left_mask
    
    G_L, H_L = np.sum(g[left_mask]), np.sum(h[left_mask])
    G_R, H_R = np.sum(g[right_mask]), np.sum(h[right_mask])
    
    sim_L = similarity_score(G_L, H_L, reg_lambda)
    sim_R = similarity_score(G_R, H_R, reg_lambda)
    sim_P = similarity_score(G_root, H_root, reg_lambda)
    
    gain = 0.5 * (sim_L + sim_R - sim_P) - gamma
    
    results.append({
        'Split Threshold': split,
        'Left Count': np.sum(left_mask),
        'Right Count': np.sum(right_mask),
        'Sim_L': round(sim_L, 2),
        'Sim_R': round(sim_R, 2),
        'Gain': round(gain, 2)
    })

res_df = pd.DataFrame(results)
print("Split Candidates & Gain (with lambda=1.0):")
print(res_df.to_string(index=False))

best_split = res_df.loc[res_df['Gain'].idxmax()]
print(f"\nBest Split Point: Area < {best_split['Split Threshold']} with Max Gain = {best_split['Gain']}")
print(f"\nNote: Only the split at 750 yields a positive gain. The others produce negative gains,")
print(f"meaning the split would actually WORSEN the objective compared to not splitting at all.")
print(f"If gamma were set higher than {best_split['Gain']}, even this split would be pruned.")

## Part 6 — Exact vs. Approximate Greedy Algorithm & Tree Pruning

### 1. Exact Greedy Algorithm

- **How it works:** For each feature, the algorithm sorts all unique values, then evaluates every consecutive midpoint as a potential split threshold by computing the Gain formula.
- **Pros:** Finds the globally optimal split for each feature.
- **Cons:** Extremely expensive for large datasets — with $n$ samples and $d$ features, sorting alone costs $O(n \\log n)$ per feature, and this must be repeated at every node during tree construction.
- **When to use:** Small to medium datasets (< ~100K rows). Set `tree_method='exact'` in XGBoost.

### 2. Approximate Greedy Algorithm (Quantile Sketch & Histogram)

For large-scale data, XGBoost provides approximate methods that dramatically reduce computation:

#### a) Weighted Quantile Sketch
- Instead of evaluating every unique value, the algorithm divides each feature into **quantile-based buckets (bins)**.
- The key innovation is that it uses the **Hessians $h_i$ as weights** when computing percentiles. This ensures that regions where the model is most uncertain (high hessian) are represented with finer granularity.
- The algorithm only evaluates split points at the bucket boundaries, reducing the number of candidates from $n$ to the number of buckets.

#### b) Histogram-based Method (`tree_method='hist'`)
- Converts continuous feature values into **discrete integer bins** (histograms) before training begins.
- Split finding operates on the histograms instead of raw data, making it extremely fast.
- This is the same core idea used by **LightGBM** and is the **recommended method** for most use cases.
- Set `tree_method='hist'` in XGBoost.

---

### 3. Tree Pruning via the $\\gamma$ Parameter

XGBoost uses a **pre-pruning** strategy controlled by $\\gamma$ (gamma, also called `min_split_loss`):

- A split is only made if the Gain exceeds $\\gamma$:
  $$\\frac{1}{2} \\left[ \\frac{G_L^2}{H_L + \\lambda} + \\frac{G_R^2}{H_R + \\lambda} - \\frac{G_P^2}{H_P + \\lambda} \\right] > \\gamma$$

- If the gain is negative (or below $\\gamma$), the branch is **pruned** — the node remains a leaf.
- **Higher $\\gamma$** → more aggressive pruning → simpler trees → less overfitting (but risk of underfitting).
- **$\\gamma = 0$** → a split is made as long as it provides any improvement at all.

> **Important Distinction:** Standard GBM typically grows trees to a fixed depth and then post-prunes. XGBoost prunes *during* construction (pre-pruning), which is more computationally efficient.

## Part 7 — Official XGBoost Library Practice (Scikit-Learn API & Native DMatrix)

Now let's move from manual calculations to the official `xgboost` library. We'll demonstrate both:

1. **Scikit-Learn compatible API** (`XGBRegressor` / `XGBClassifier`) — familiar interface for sklearn users
2. **Native XGBoost API** (`xgb.train` with `DMatrix`) — more control, supports advanced features like custom objectives and watchlists

### 7.1 — Regression with the Scikit-Learn API

In [ ]:
import xgboost as xgb
from sklearn.datasets import make_classification, make_regression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, accuracy_score, roc_auc_score

# 1. Regression Practice
# Generate a synthetic regression dataset with 500 samples and 10 features
X_reg, y_reg = make_regression(n_samples=500, n_features=10, noise=0.1, random_state=42)
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

# XGBoost Regressor using the Scikit-Learn compatible API
model_reg = xgb.XGBRegressor(
    n_estimators=100,       # Number of boosting rounds (trees)
    learning_rate=0.05,     # Shrinkage factor (eta) — smaller = slower learning, better generalization
    max_depth=4,            # Maximum depth of each tree
    gamma=0.1,              # Minimum loss reduction required to make a split
    reg_alpha=0.1,          # L1 regularization on leaf weights
    reg_lambda=1.0,         # L2 regularization on leaf weights
    tree_method='hist',     # Fast histogram-based split finding
    random_state=42
)

model_reg.fit(X_train_r, y_train_r)
preds_r = model_reg.predict(X_test_r)
rmse = np.sqrt(mean_squared_error(y_test_r, preds_r))
print(f"XGBRegressor RMSE: {rmse:.4f}")

### 7.2 — Binary Classification with the Native DMatrix API

The **DMatrix** is XGBoost's internal, optimized data structure. It stores data in a compressed, column-oriented format that enables fast, cache-friendly access during tree building. Using the native API gives you:
- Fine-grained control over training (watchlists, custom callbacks)
- The ability to use features like `early_stopping_rounds`
- Access to the full set of XGBoost parameters

In [ ]:
# 2. Classification Practice with the Native DMatrix API
# Generate a synthetic binary classification dataset
X_cls, y_cls = make_classification(n_samples=1000, n_features=12, n_informative=8, random_state=42)
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_cls, y_cls, test_size=0.2, random_state=42)

# Convert to DMatrix format (XGBoost's internal optimized data structure)
dtrain = xgb.DMatrix(X_train_c, label=y_train_c)
dtest = xgb.DMatrix(X_test_c, label=y_test_c)

# Define hyperparameters as a dictionary
params = {
    'objective': 'binary:logistic',  # Binary classification with logistic loss
    'eval_metric': 'logloss',        # Evaluation metric for the watchlist
    'max_depth': 5,                  # Maximum tree depth
    'eta': 0.1,                      # Learning rate (same as learning_rate in sklearn API)
    'gamma': 0.2,                    # Split pruning threshold
    'lambda': 1.5,                   # L2 regularization
    'alpha': 0.5,                    # L1 regularization
    'subsample': 0.8,               # Row subsampling ratio (stochastic gradient boosting)
    'colsample_bytree': 0.8,        # Column subsampling ratio per tree (feature bagging)
    'tree_method': 'hist'            # Histogram-based split finding
}

# Watchlist: monitors performance on both train and test sets during training
evals = [(dtrain, 'train'), (dtest, 'eval')]
bst = xgb.train(
    params,
    dtrain,
    num_boost_round=150,    # Total number of boosting iterations
    evals=evals,            # Watchlist for monitoring
    verbose_eval=30         # Print evaluation metrics every 30 rounds
)

# Predict probabilities on the test set
probs = bst.predict(dtest)
preds = (probs > 0.5).astype(int)  # Convert probabilities to binary predictions

acc = accuracy_score(y_test_c, preds)
auc = roc_auc_score(y_test_c, probs)
print(f"\nNative XGBoost Accuracy: {acc:.4f}")
print(f"Native XGBoost ROC-AUC: {auc:.4f}")

## Part 8 — Feature Importance & Tree Visualization

XGBoost provides built-in feature importance scores. The default metric is **weight** (the number of times a feature is used as a split across all trees). Other options include:
- `importance_type='gain'` — average gain contributed by the feature when it is used for splitting
- `importance_type='cover'` — average number of samples affected by splits on this feature

In [ ]:
# Plot Feature Importance (by weight — number of times each feature is used in splits)
fig, ax = plt.subplots(figsize=(8, 5))
xgb.plot_importance(bst, max_num_features=10, ax=ax, title="XGBoost Feature Importance (Weight)")
plt.tight_layout()
plt.show()

## Part 9 — Summary & Key Takeaways

Let's consolidate everything we've learned about XGBoost's architecture and usage:

1. **2nd-Order Taylor Expansion:** Unlike standard GBM (which only uses 1st-order gradients), XGBoost uses both the gradient $g_i$ and the hessian $h_i$ to build a quadratic approximation of the loss. This allows it to derive the exact optimal leaf weight $w^* = -G / (H + \\lambda)$ in closed form.

2. **Built-in Regularization:** The L2 parameter $\\lambda$ shrinks leaf weights toward zero (smoothing predictions), while the L1 parameter $\\alpha$ encourages sparsity. Together, they prevent overfitting far more effectively than learning rate alone.

3. **Similarity Score & Gain:** The Similarity Score $G^2 / (H + \\lambda)$ measures the quality of a node. The Gain formula quantifies how much a proposed split improves the objective. The $\\gamma$ parameter sets a minimum gain threshold — any split below this threshold is pruned.

4. **Exact vs. Approximate Greedy Split Finding:** For small datasets, the exact method (`tree_method='exact'`) evaluates every possible split. For large datasets, the histogram method (`tree_method='hist'`) bins features into discrete buckets for much faster split finding.

5. **Key Hyperparameters to Tune:**
   - `learning_rate` ($\\eta$): Controls the contribution of each tree. Lower values require more trees but often generalize better.
   - `max_depth`: Controls tree complexity. Typical values: 3–10.
   - `gamma` ($\\gamma$): Minimum gain for a split. Higher values → more pruning.
   - `reg_alpha` ($\\alpha$): L1 regularization on leaf weights.
   - `reg_lambda` ($\\lambda$): L2 regularization on leaf weights.
   - `subsample`: Fraction of rows sampled per tree (stochastic boosting).
   - `colsample_bytree`: Fraction of features sampled per tree (feature bagging).

6. **Two APIs:**
   - **Scikit-Learn API** (`XGBRegressor`, `XGBClassifier`): Easy integration with sklearn pipelines, cross-validation, and grid search.
   - **Native API** (`xgb.train` + `DMatrix`): More control, custom objectives, watchlists, and callbacks.

---

> **Next Steps:** Practice tuning these hyperparameters on real-world datasets using cross-validation (`xgb.cv`) or sklearn's `GridSearchCV` / `RandomizedSearchCV`. Experiment with `early_stopping_rounds` to find the optimal number of boosting iterations automatically.